# Sham — CPU Training Track (self-schedulable, no GPU quota)

**Why this notebook exists (owner, 2026-09-20):** the main training notebook is capped by Kaggle's real, documented **30 GPU-hours/week** quota. CPU compute is **not** subject to that cap — a CPU-only notebook can run far more often, accumulating real training hours the GPU track structurally can't. It trains much more slowly per step (dense transformer training on CPU vs. a T4 GPU is a real, large slowdown — expect roughly one to two orders of magnitude fewer steps/second, not a small difference), which is exactly why this is framed as "slow but continuous" rather than a GPU replacement.

**Because this notebook has NO accelerator, Kaggle's own native "Schedule this notebook" feature accepts it directly** (only GPU-attached notebooks are refused) — no separate orchestrator notebook is needed for this one, unlike the GPU training notebook.

**Separate checkpoint lineage:** this track publishes only to its own dataset `sham-cpu-track-checkpoint-v2`, never into the GPU track's `sham-checkpoint` — two independently scheduled processes writing the same weights would race.

**Resume is automatic (2026-09-24 fix):** every run fetches the latest version of `sham-cpu-track-checkpoint-v2` itself through the Kaggle API (`sham_inputs.py`) and continues from its highest step — no "Add Input" needed, and a stale Input (the old name without `-v2`, which silently lost progress) can no longer win. First run ever: one-time fork from the GPU track's `sham-checkpoint`, with that dataset's own tokenizer. Wikipedia reading also continues from the saved position each run instead of the same first 5,000 articles.

**One-time setup checklist:**
1. Accelerator: **None**. Internet: **On**.
2. Secrets (Add-ons): `GITHUB_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY` (+ optional `TELEGRAM_BOT_TOKEN`, `TELEGRAM_CHAT_ID`).
3. Save & Run All, then Kaggle's own **"Schedule this notebook"** (or leave it to the resume orchestrator).

### 1) Clone the real code from GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py"))
sys.path.insert(0, CODE_DIR)
print("Sham code ready at:", CODE_DIR)

In [ ]:
try:
    import tokenizers
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
print("tokenizers ready.")

### 1b) Resume point — fetched automatically (no manual Inputs)

In [ ]:
from sham_inputs import resume_text_lineage

# Own lineage first (attached or downloaded); only on the very first run, a one-time fork
# from the GPU track. Old "sham-cpu-track-checkpoint" (no -v2) is a last-resort fork only.
LINEAGE = resume_text_lineage(
    "sham-cpu-track-checkpoint-v2",
    forks=["sham-cpu-track-checkpoint", "sham-checkpoint"],
)

### 2) Real Arabic text data (same streaming source as the GPU track)

In [ ]:
from data_acquisition import stream_hf_text_corpus, text_stream_position

MAX_DOCUMENTS = 5_000
WIKI = ("wikimedia/wikipedia", "20231101.ar")
# كل تشغيل يكمل من حيث توقّف السابق في ويكيبيديا بدل إعادة نفس أول المقالات إلى الأبد:
# الموضع محفوظ في text_stream_progress_cpu.json داخل مجموعة بيانات هذا المسار. التشغيلات القديمة قبل هذا
# الإصلاح لم تحفظ موضعاً وكانت تقرأ أول 20,000 مقالة دائماً، فنبدأ بعدها.
stream_skip = text_stream_position(*WIKI) or 20_000
corpus_dir = "/kaggle/working/corpus/wikipedia_ar"
_progress = "/kaggle/working/checkpoints/text_stream_progress_cpu.json"
corpus_files = stream_hf_text_corpus(
    dataset_name=WIKI[0], config_name=WIKI[1], text_field="text", output_dir=corpus_dir,
    max_documents=MAX_DOCUMENTS, skip=stream_skip, progress_path=_progress,
)
if not corpus_files:  # وصلنا لنهاية ويكيبيديا العربية كلها — دورة جديدة من البداية
    corpus_files = stream_hf_text_corpus(
        dataset_name=WIKI[0], config_name=WIKI[1], text_field="text", output_dir=corpus_dir,
        max_documents=MAX_DOCUMENTS, skip=0, progress_path=_progress,
    )
print(f"{len(corpus_files)} shard file(s) from article #{stream_skip:,} onward")

### 3) Tokenizer — MUST match the GPU track's vocab for the one-time fork to load correctly

If resuming from this track's own prior checkpoint, its own saved tokenizer is reused. On the very first run (forking from the GPU track), reuse the GPU track's own saved tokenizer file (attached via its dataset Input) instead of training a fresh one -- a mismatched vocabulary would silently corrupt every forked weight's meaning.

In [ ]:
from pathlib import Path
from text_tokenizer import train_text_tokenizer, ShamTextTokenizer
from model import TEXT_VOCAB_SIZE

# Always the tokenizer saved WITH the checkpoint being resumed -- a mismatched vocabulary
# would silently corrupt every weight's meaning.
if LINEAGE:
    tokenizer = ShamTextTokenizer.load(str(LINEAGE["tokenizer"]))
    print(f"reused the tokenizer saved with the resumed checkpoint: {LINEAGE['tokenizer']}")
else:
    tokenizer = train_text_tokenizer(corpus_files, vocab_size=TEXT_VOCAB_SIZE)
    print("trained a fresh tokenizer (no previous one found -- this should only happen on a true first run).")
tokenizer.save("/kaggle/working/sham_cpu_tokenizer.json")

### 4) Build training windows

In [ ]:
import torch
from dataset import TextSequenceDataset

SEQ_LEN = 512  # shorter than the GPU track's 1024 -- CPU attention cost grows with seq_len^2, and this track's whole point is real completed steps, not a long context per step
text_dataset = TextSequenceDataset(corpus_files, tokenizer, seq_len=SEQ_LEN)
print(f"real training windows: {len(text_dataset):,}")

### 5) Model size -- MUST match the GPU track's "starter" config exactly for the one-time fork's weights to load

In [ ]:
from model import ShamSmallConfig, ShamSmall, TOTAL_VOCAB_SIZE

model_cfg = ShamSmallConfig(
    vocab_size=TOTAL_VOCAB_SIZE, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4,
    mlp_hidden=2048, max_seq_len=SEQ_LEN, use_gradient_checkpointing=True,
)
model = ShamSmall(model_cfg)
print(f"model: {model.count_parameters():,} real parameters.")

device = "cpu"  # this track is CPU-only by design; no accelerator is ever attached to this notebook
print(f"device: {device}")

### 6) Resume -- from THIS track's own prior checkpoint, or the GPU track's (one-time fork only)

Looks across every attached Input, exactly like the GPU notebook's own resume cell -- the safety here comes from the setup checklist above (only the GPU dataset is attached on the very first run, then removed), not from any automatic lineage detection in this cell.

In [ ]:
from checkpoint import load_checkpoint

start_step = 0
resume_optimizer = None
if LINEAGE:
    model, start_step, _ = load_checkpoint(LINEAGE["checkpoint"], map_location=device)
    if LINEAGE["has_optimizer"]:
        from train import build_optimizer
        resume_optimizer = build_optimizer(model, lr=3e-4, weight_decay=0.1)
        load_checkpoint(LINEAGE["checkpoint"], map_location=device, load_optimizer_into=resume_optimizer)
    print(f"resumed from {LINEAGE['dataset']}/{LINEAGE['checkpoint'].name} (step {start_step:,})")
else:
    print("no previous checkpoint found -- starting from scratch (expected only on a true first run).")

### 7) Measure real CPU speed, then train for a bounded wall-clock budget

In [ ]:
import time
from train import TrainConfig, build_optimizer, build_lr_scheduler

# Warm-up steps run for real but are NOT timed -- the very first real steps
# on any device include one-off costs (memory allocation, first-touch page
# faults, cache warm-up) that make measuring from step 1 pessimistic. Real
# incident (owner-observed, 2026-09-21, on this same calibration pattern in
# the GPU main track): calibrating on the first steps only produced a step
# budget so conservative the session finished it in a fraction of
# MAX_TRAINING_HOURS and stopped cleanly, wasting most of the available
# session time. Measuring after warm-up gives a realistic sustained-speed
# estimate instead.
CALIBRATION_WARMUP_STEPS = 5
CALIBRATION_STEPS = 30

calib_batches = [
    torch.stack([text_dataset[i] for i in range(b, b + 2)])
    for b in range(0, min(len(text_dataset) - 2, (CALIBRATION_WARMUP_STEPS + CALIBRATION_STEPS) * 2 * 4), 2)
][: (CALIBRATION_WARMUP_STEPS + CALIBRATION_STEPS) * 4]
assert len(calib_batches) > CALIBRATION_WARMUP_STEPS, "not enough data to calibrate -- increase MAX_DOCUMENTS above."

model.to(device)
model.train()
_calib_optimizer = resume_optimizer or build_optimizer(model, lr=3e-4, weight_decay=0.1)

def _calib_step(batch):
    batch = batch.to(device)
    _, loss = model(batch, labels=batch)
    loss.backward()
    _calib_optimizer.step()
    _calib_optimizer.zero_grad()

for batch in calib_batches[:CALIBRATION_WARMUP_STEPS]:
    _calib_step(batch)

t0 = time.time()
steps_done = 0
for batch in calib_batches[CALIBRATION_WARMUP_STEPS:]:
    _calib_step(batch)
    steps_done += 1
    if steps_done >= CALIBRATION_STEPS:
        break
elapsed = time.time() - t0
steps_per_second = steps_done / elapsed

# Not necessarily the same limit as GPU sessions -- adjust if your own
# observed CPU session length on Kaggle differs from this conservative default.
MAX_TRAINING_HOURS = 8.5
realistic_steps_for_session = max(int(steps_per_second * MAX_TRAINING_HOURS * 3600 * 0.85), 20)

print(f"real measured speed (after {CALIBRATION_WARMUP_STEPS} warm-up steps): {steps_per_second:.4f} steps/sec on {device}")
print(f"realistic steps for this session (with safety margin): {realistic_steps_for_session:,}")


### 8) Train

In [ ]:
TOTAL_STEPS = realistic_steps_for_session
num_windows = len(text_dataset) - (len(text_dataset) % 4)

def _batch_iterator():
    while True:
        for b in range(0, num_windows, 4):
            yield torch.stack([text_dataset[i] for i in range(b, b + 4)])

import itertools
batches = itertools.islice(_batch_iterator(), TOTAL_STEPS)

train_cfg = TrainConfig(
    seq_len=SEQ_LEN,
    batch_size=4,
    grad_accum_steps=4,
    lr=3e-4,
    warmup_steps=max(20, TOTAL_STEPS // 100),
    total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints",
    checkpoint_every=100,
    log_every=10,
    max_wall_clock_seconds=MAX_TRAINING_HOURS * 3600,
)

from train import train
loss_history = train(
    model, batches, train_cfg, device=device,
    start_step=start_step, resume_optimizer=resume_optimizer or _calib_optimizer,
)
print(f"\nreal steps this session: {len(loss_history):,}")
if loss_history:
    print(f"first 10 avg loss: {sum(loss_history[:10]) / min(10, len(loss_history)):.4f}")
    print(f"last 10 avg loss: {sum(loss_history[-10:]) / min(10, len(loss_history)):.4f}")

### 9) Save + publish this track's own checkpoint (its own dataset, never the GPU track's)

In [ ]:
from checkpoint import save_checkpoint

final_step = start_step + len(loss_history)
save_checkpoint("/kaggle/working/checkpoints/final.pt", model, final_step)
print(f"final checkpoint saved locally at step {final_step:,}.")

In [ ]:
import json as _json
import shutil as _shutil

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=False)

KAGGLE_USERNAME = UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")
# "-v2": the original "sham-cpu-track-checkpoint" was born via a manual
# placeholder-file upload on Kaggle's website, an unknown quantity for
# whether it would accept a later "-r zip" version push. Rather than find
# out empirically mid-run (as happened on the GPU track's identical
# dataset), this track starts on a name Kaggle has never seen before, so
# its very first version establishes zip-mode compatibility from birth.
DATASET_SLUG = f"{KAGGLE_USERNAME}/sham-cpu-track-checkpoint-v2"

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    _shutil.rmtree(upload_dir)
(upload_dir / "checkpoints").mkdir(parents=True)
for ckpt in Path("/kaggle/working/checkpoints").glob("*.pt"):
    _shutil.copy2(ckpt, upload_dir / "checkpoints" / ckpt.name)
_shutil.copy2("/kaggle/working/sham_cpu_tokenizer.json", upload_dir / "sham_cpu_tokenizer.json")
for _extra in Path("/kaggle/working/checkpoints").glob("*.json"):  # موضع القراءة من ويكيبيديا
    _shutil.copy2(_extra, upload_dir / _extra.name)

metadata = {"title": "sham-cpu-track-checkpoint-v2", "id": DATASET_SLUG, "licenses": [{"name": "unknown"}]}
(upload_dir / "dataset-metadata.json").write_text(_json.dumps(metadata))

# Deterministic existence check -- ask Kaggle directly whether this
# dataset is already in the account's own list, instead of guessing from
# an error message's wording (a version-command failure's exact text
# isn't a documented, stable contract, so string-matching it is fragile
# by nature -- exactly the kind of fragility that caused this bug to go
# unnoticed for a long time in the first place).
_list_result = subprocess.run(["kaggle", "datasets", "list", "-m", "--csv"], capture_output=True, text=True)
_dataset_exists = DATASET_SLUG in (_list_result.stdout or "")

# "-r zip" (not "skip"): "-r skip" is Kaggle CLI's documented default for
# subdirectories -- it silently IGNORES them instead of uploading them.
# checkpoints/ is a subdirectory, so a version published under "-r skip"
# never actually contains the .pt files, only the flat tokenizer file
# alongside it -- despite printing success (owner-caught on the GPU
# track's identical bug, 2026-09-20). "-r zip" actually uploads the
# subdirectory, as a .zip archive.
if _dataset_exists:
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", f"cpu-track auto-update at step {final_step:,}", "-r", "zip"],
        capture_output=True, text=True,
    )
else:
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(upload_dir), "-r", "zip"],
        capture_output=True, text=True,
    )
_combined = (result.stdout or "") + (result.stderr or "")

if result.returncode == 0 and "error" not in _combined.lower():
    verb = "published to" if _dataset_exists else "created"
    print(f"{verb} {DATASET_SLUG} at step {final_step:,} -- the next scheduled run will pick it up automatically.")
elif "incompatible" in _combined.lower():
    # Should not happen on a dataset this cell itself just created, but
    # kept as a clear, actionable message rather than a silent dead end
    # in case Kaggle's rules around this ever surprise us again.
    print(
        "WARNING: this dataset was created under an incompatible upload mode. No code-level fix for it "
        "specifically -- bump the -v2 suffix above to -v3 (a name Kaggle has never seen) and re-run. "
        "The checkpoint itself is still safe in this session's own Output regardless."
    )
    print(_combined)
else:
    print("WARNING: failed to publish -- the checkpoint is still safe in this session's own Output. "
          "Check that KAGGLE_USERNAME/KAGGLE_KEY are correct.")
    print(_combined)

### 10) Report this session to the owner on Telegram (optional)

Same idea as Nova's automatic weekly Telegram report, applied to this track's own training sessions -- see `telegram_report.py`. Skips cleanly if `TELEGRAM_BOT_TOKEN`/`TELEGRAM_CHAT_ID` secrets aren't configured (see the cell below for one-time setup).

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    TELEGRAM_BOT_TOKEN = UserSecretsClient().get_secret("TELEGRAM_BOT_TOKEN")
    TELEGRAM_CHAT_ID = UserSecretsClient().get_secret("TELEGRAM_CHAT_ID")
except Exception:
    # Both secrets are OPTIONAL -- a real, one-time setup step (owner
    # request, 2026-09-21, comparing Nova's automatic Telegram reports
    # against Sham's total silence between manual Kaggle-log checks).
    # Add-ons -> Secrets -> add TELEGRAM_BOT_TOKEN (an existing bot's
    # token) and TELEGRAM_CHAT_ID (the owner's own Telegram numeric id,
    # e.g. the SUPER_ADMIN_TELEGRAM_ID already used elsewhere in this
    # project) to receive this. Not configured -> report is skipped,
    # never blocks or fails the session itself.
    TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID = None, None

from telegram_report import format_training_report, send_telegram_message

_published_slug = DATASET_SLUG if (result.returncode == 0 and "error" not in _combined.lower()) else None
_report_text = format_training_report("التدريب المستمر على المعالج المركزي (Track A)", final_step, loss_history, _published_slug)
print(_report_text)
send_telegram_message(TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID, _report_text)
